In [1]:
import pandas as pd

gsv_wide = pd.read_csv("data/gsv_factors.csv")
gsv_wide

,policy_year,pt_10,pt_12,pt_15,pt_20
0,1,NaN,NaN,NaN,NaN
1,2,30.0,30.0,30.0,30.0
2,3,35.0,35.0,35.0,35.0
3,4,50.0,50.0,50.0,50.0
4,5,50.0,50.0,50.0,50.0
5,6,50.0,50.0,50.0,50.0
6,7,50.0,50.0,50.0,50.0
7,8,70.0,60.0,56.0,53.0
8,9,90.0,70.0,61.0,57.0
9,10,90.0,80.0,67.0,60.0


In [2]:
gsv_wide.shape # (20, 5) means 20 rows and 5 columns
gsv_wide.dtypes #the data type of each column

policy_year      int64
pt_10          float64
pt_12          float64
pt_15          float64
pt_20          float64
dtype: object

In [3]:
gsv = gsv_wide.melt(
    id_vars = "policy_year", #the column that stays as it is
    var_name = "pt", #the old column names go into this new column
    value_name = "gsv_pct" #the cell values go into this new column
)
gsv.head(8)

,policy_year,pt,gsv_pct
0,1,pt_10,NaN
1,2,pt_10,30.0
2,3,pt_10,35.0
3,4,pt_10,50.0
4,5,pt_10,50.0
5,6,pt_10,50.0
6,7,pt_10,50.0
7,8,pt_10,70.0


In [4]:
gsv["pt"] = gsv["pt"].str.replace("pt_", "").astype(int)

#1 Keep only years that exist within each policy term
gsv = gsv[gsv["policy_year"] <= gsv["pt"]].copy()

#2 Year 1 shows "-" in the bond: no GSV, so the factor is 0 
gsv.loc[gsv["policy_year"] == 1, "gsv_pct"] = 0

#3 Nothing else should be empty now. If it is, the CSV has a typo.
assert gsv["gsv_pct"].notna().all()

gsv["gsv_factor"] = gsv["gsv_pct"] / 100
gsv.groupby("pt").size()  # now 10, 12, 15, 20

pt
10    10
12    12
15    15
20    20
dtype: int64

In [5]:
gsv_lookup = gsv.set_index(["pt", "policy_year"])["gsv_factor"]

gsv_lookup.loc[(15, 10)] #PT 15, year 10, should give 0.67
gsv_lookup.loc[(20, 12)] #PT 20, year 12, should give 0.67

np.float64(0.67)

In [7]:
#1. Row count per policy term 
print(gsv.groupby("pt").size())

#2. Every factor should lie between 0 and 0.90
assert gsv["gsv_factor"].between(0, 0.90).all()

#3. Within a policy term, the factor should never fall as the years go on
for term, g in gsv.groupby("pt"):
    assert g.sort_values("policy_year")["gsv_factor"].is_monotonic_increasing, term

#4. The last year of each term should be 0.90
assert (gsv[gsv["policy_year"] == gsv["pt"]]["gsv_factor"] == 0.90).all()

pt
10    10
12    12
15    15
20    20
dtype: int64


In [8]:
gsv.query("pt == 12 and policy_year >= 8")

,policy_year,pt,gsv_pct,gsv_factor
27,8,12,60.0,0.6
28,9,12,70.0,0.7
29,10,12,80.0,0.8
30,11,12,90.0,0.9
31,12,12,90.0,0.9
